In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from collections import defaultdict

#custom functions
from heuristic_functions import *

##### Assumptions
- 1 year of a tender is allocated 1 year of antigen demand (through 1 or many vaccines); scales appropriately
- Intenvory is based on demand for a given year.
- Ratio of inventory to supply is calculated based on end of year totals (after calcs), but only in the programatic sense (check after math, add inventory at end of year for next year)

##### Load and setup demand

In [2]:
# Load the CSV file
demand_path = 'data/real/antigen_demand_80_20_2_scenarios.csv'
data = pd.read_csv(demand_path)
# Create the two dataframes based on the 'prob' column
demand_80 = data[data['prob'] == 0.8]
demand_20 = data[data['prob'] == 0.2]

demand_80 = demand_80.drop(columns=['prob', 'demand_SID'])
demand_20 = demand_20.drop(columns=['prob', 'demand_SID'])
# Expanding the 'demands' column into 10 separate columns
demand_80_expanded = demand_80['demands'].apply(lambda x: pd.Series(eval(x)))
demand_20_expanded = demand_20['demands'].apply(lambda x: pd.Series(eval(x)))

# Renaming the columns to 1-10
demand_80_expanded.columns = range(1, 11)
demand_20_expanded.columns = range(1, 11)

# Concatenating the expanded demands columns back to the original antigen column
demand_80_final = pd.concat([demand_80['antigen'], demand_80_expanded], axis=1)
#added 11th year to capture any left overdemand at the end of year 10.
demand_80_final[11] = 0.1
demand_20_final = pd.concat([demand_20['antigen'], demand_20_expanded], axis=1)
#added 11th year to capture any left overdemand at the end of year 10.
demand_20_final[11] = 0.1

# demand_80_final.head(), demand_20_final.head()


##### Load and setup Starting Points

In [3]:
file_path = 'data/real/Starting_point.xlsx'

# Load the sheets 'F_start', 'I_start', 'S_start' into their own DataFrames
f_start = pd.read_excel(file_path, sheet_name='F_start')
i_start = pd.read_excel(file_path, sheet_name='I_start')
s_start = pd.read_excel(file_path, sheet_name='S_start')

##### Load pricing data

In [4]:
# Load the Excel file, skipping the first two sheets
money_path = 'data/Vaccine_price_data.xlsx'
sheet_names = pd.ExcelFile(money_path).sheet_names

# Load the remaining sheets into a dictionary of DataFrames
# data = {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in sheet_names[2:5]}
price_data = {sheet_names[i]: pd.read_excel(money_path, sheet_name=i).rename(columns=lambda x: "Manufacturer" if x == pd.read_excel(money_path, sheet_name=i).columns[0] else x) for i in range(2, 5)}


##### Load capacity data

In [5]:
# Load the Excel file, only reading the first sheet
capacity_path = 'data/production_capacity_scenarios.xlsx'
sheet_names = pd.ExcelFile(capacity_path).sheet_names

capacity_data = pd.read_excel(capacity_path, sheet_name='base_capacity')
# capacity_data

##### Initialize stuff

In [6]:
# Creating an empty DataFrame with the specified structure for calculating ratios
antigens = f_start['Antigen']
columns = ['Antigen',1]

ratio_DF = pd.DataFrame(columns=columns)
ratio_DF['Antigen'] = antigens
ratio_DF[1] = np.zeros(len(antigens))

# create DF to store tender schedule
tender_schedules = f_start.copy()

########################################################
#create DF to store current inventory
inventory_DF = i_start.copy()
##########################################################
#create DF to store missed doses
antigens = f_start['Antigen']
columns = ['Antigen','Missed Doses']

missed_doses = pd.DataFrame(columns=columns)
missed_doses['Antigen'] = antigens
missed_doses['Missed Doses'] = np.zeros(len(antigens))

#CREATE TO STORE VACCINES "PURCHASED" THROUGH TENDERS
vaccine_purchases = defaultdict(list)

#track total price throughout
# update for tenders, vaccine data
total_price = 0

#tender cost
tender_cost = 10000


##### Initialize antigens/vaccines/prodicers

In [7]:
# A = ["Measles", "Mumps", "Rubella"]
# V = ["M", "MR", "MMR"]

# A_v = {
#     "M": ["Measles"],
#     "MR": ["Measles", "Rubella"],
#     "MMR": ["Measles", "Mumps", "Rubella"]
# }

# P = ["Biological_E", 
#     "GSK","PT_Bio", 
#     "Serum_Institute"
# ]

# P_v = {
#     "M": ["Serum_Institute", "PT_Bio"],
#     "MR": ["Serum_Institute", "Biological_E"],
#     "MMR": ["Serum_Institute", "GSK"]
# }

A = ["Measles", "Mumps", "Rubella", "Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio", "HPV", "Rotavirus", "PCV"]

V = ["M", "MR", "MMR", "TT", "HepB", "Hib", "IPV", "OPV", "DT", "Td", "DTwP", "DTwP-Hib", "Penta", "Hexa", "HPV", "Rotavirus", "PCV"]

A_v = {
    "M": ["Measles"],
    "MR": ["Measles", "Rubella"],
    "MMR": ["Measles", "Mumps", "Rubella"],
    "TT": ["Tetanus"],
    "HepB": ["Hepatitis_B"],
    "Hib": ["Hib"],
    "IPV": ["Polio"],
    "OPV": ["Polio"],
    "DT": ["Diphtheria", "Tetanus"],
    "Td": ["Diphtheria", "Tetanus"],
    "DTwP": ["Diphtheria", "Tetanus", "Pertussis"],
    "DTwP-Hib": ["Diphtheria", "Tetanus", "Pertussis", "Hib"],
    "Penta": ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib"],
    "Hexa": ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio"],
    "HPV": ["HPV"],
    "Rotavirus": ["Rotavirus"],
    "PCV": ["PCV"]
}

P = ["AJ_Vaccines", "BB_NCIPD", "China_National", "Bharat_Biotech", "Bilthoven", "Biological_E", "GSK", "Haffkine_Bio",
     "LG_Chem", "Merck_Sharp", "Panacea_Biotec", "PT_Bio", "Sanofi", "Serum_Institute", "Pfizer"]

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"],
    "TT": ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"],
    "HepB": ["Serum_Institute", "LG_Chem"],
    "Hib": ["Serum_Institute"],
    "IPV": ["LG_Chem", "AJ_Vaccines", "Bilthoven", "Sanofi"],
    "OPV": ["Serum_Institute", "PT_Bio", "GSK", "Sanofi", "Panacea_Biotec", "China_National", "Bharat_Biotech", "Haffkine_Bio"],
    "DT": ["PT_Bio", "BB_NCIPD"],
    "Td": ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"],
    "DTwP": ["Serum_Institute", "Biological_E"],
    "DTwP-Hib": ["Serum_Institute"],
    "Penta": ["Serum_Institute", "PT_Bio", "Biological_E", "LG_Chem", "Panacea_Biotec"],
    "Hexa": ["Sanofi"],
    "HPV": ["GSK", "Merck_Sharp", "China_National"],
    "Rotavirus": ["Serum_Institute", "GSK", "Bharat_Biotech"],
    "PCV": ["Serum_Institute", "GSK", "Pfizer"]
}


#translate vaccine - antigen, to antigen - vaccine
V_a = {a: [v for v in A_v if a in A_v[v]] for a in A}

V_p = {p: [v for v in P_v if p in P_v[v]] for p in P}

P_a = {a: list(set(p for v in V_a[a] for p in P_v[v])) for a in A}

A_p = {p: [a for a in P_a if p in P_a[a]] for p in P}



## TESTING - Measles Containing Vaccines Only

In [8]:
# # Selecting only the rows for 'Measles', 'Mumps', and 'Rubella' in both datasets
# demand_80_MCV = demand_80_final[demand_80_final['antigen'].isin(['Measles', 'Mumps', 'Rubella'])]

# interim_demand_DF = demand_80_MCV.copy()
# tender_schedules_MCV = tender_schedules[tender_schedules['Antigen'].isin(['Measles', 'Mumps', 'Rubella'])]


# missed_doses_MCV = missed_doses[missed_doses['Antigen'].isin(['Measles', 'Mumps', 'Rubella'])]

# ratio_DF_MCV = pd.DataFrame()
# inventory_DF_MCV = inventory_DF[inventory_DF['Vaccine'].isin(['M', 'MR', 'MMR'])]



In [9]:
# Selecting only the rows for 'Measles', 'Mumps', and 'Rubella' in both datasets

demand_80_MCV = demand_80_final
interim_demand_DF = demand_80_MCV.copy()

tender_schedules_MCV = tender_schedules
i_start_MCV = i_start

missed_doses_MCV = missed_doses
ratio_DF_MCV = pd.DataFrame()

inventory_DF_MCV = inventory_DF

In [10]:
inventory_DF_MCV

,Vaccine,Amount
0,Penta,1.370105e+08
1,OPV,2.837175e+08
2,IPV,9.879500e+06
3,PCV,1.088040e+08
4,M,2.575814e+08
5,MR,8.375001e+08
6,MMR,7.846400e+07
7,TT,1.000000e+07
8,HepB,1.000000e+07
9,Hib,1.000000e+07


In [11]:
demand_80_MCV

,antigen,1,2,3,4,5,6,7,8,9,10,11
0,Diphtheria,570550200,619539900,626455400,594496900,595980200,587103300,593084700,587954100,594707400,600159000,0.1
2,HPV,18988200,18779800,27460000,72849500,92670600,64703800,42237000,42368400,33730500,35038500,0.1
4,Hepatitis_B,312284600,343008200,351717800,334973600,334455600,328430100,332308900,328200500,332214900,335525400,0.1
6,Hib,259536200,282777100,287252900,274027900,274339400,270041300,273119600,270415600,273713400,276715500,0.1
8,Polio,522182000,582182300,608631800,590689600,591743500,572503000,130824100,128516200,130807300,132956000,0.1
10,Measles,420097000,358616900,394831600,346093200,377569100,407486200,271512200,386591100,290515900,326808200,0.1
12,Mumps,26052200,26626400,25785400,25268100,23536200,16205400,18650400,8909600,10718500,12977700,0.1
14,PCV,160800200,186996100,209165500,205938500,220998700,219106600,222271100,217335600,218764600,220234300,0.1
16,Pertussis,320662700,347414000,352464600,335977000,334855300,328035800,331237500,328308900,331462200,334312000,0.1
18,Rotavirus,142479000,166422100,173751400,186328400,197668300,205652300,209552800,207355600,209606300,211802800,0.1


##### Logic to translate vaccine totals to antigen coverage for later math

In [12]:
#logic to setup least covered antigens:
# Flatten the list of all antigens from all vaccines
all_antigens = [antigen for antigens in A_v.values() for antigen in antigens]
# Count the occurrences of each antigen
antigen_counts = Counter(all_antigens)
# least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)
tender_length = 3

In [13]:
for year in range(1, 11):  # Iterate through each year - short range for testing  range(1,len(demand_80_MCV.columns)-1)
    print("********************HAPPY NEW YEAR****************************")
    print("******************Reticulating splines...**************************")
    print(f"Year: {year}")

    least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)
    remaining_inventory = {}
    uncovered_demand = {}
    print(f"least covered antigens: {least_covered_antigens}")
    while least_covered_antigens:  # Iterate through each antigen, find what vaccines cover each antigen, least to greatest, update supply and demand
        print('###############################################################')
        antigen = least_covered_antigens.pop(0)
        print(f"serving antigen {antigen}")
        print(f"Initial Demand: {demand_80_MCV.loc[demand_80_MCV['antigen'] == antigen, year].iloc[0]} for {antigen}")
        print(f"Initial Inventory of vaccine covering {antigen}: {inventory_DF_MCV}")
        print(f"A_v items are: {A_v.items()}")
        for vaccine, antigens in A_v.items():  # Iterate through A_v to check which vaccines cover the antigen
            print(f"antigens list is: {antigens}")
            if antigen in antigens and demand_80_MCV.loc[demand_80_MCV['antigen'] == antigen, year].iloc[0] >0:
                print(f"{antigen} FOUND in {vaccine}")
                vaccine_inventory_value = inventory_DF_MCV.loc[inventory_DF_MCV['Vaccine'] == vaccine, 'Amount'].iloc[0]
                print("iiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii")
                print(f"Inventory for {vaccine} for year {year}: ", vaccine_inventory_value)

                antigen_demand_value = demand_80_MCV.loc[demand_80_MCV['antigen'] == antigen, year].iloc[0]
                print(f"Demand value for {antigen} for year {year}: ", antigen_demand_value)

                difference = vaccine_inventory_value - antigen_demand_value
                print(f"The difference between supply and demand is : {difference}")
                if difference >= 0: #
                    remaining_inventory[vaccine] = difference
                    decrement = antigen_demand_value
                else: 
                    remaining_inventory[vaccine] = 0
                    decrement = vaccine_inventory_value
                    uncovered_demand[antigen] = abs(difference)
                    # print("------------------------------------------")
                    # print(f"Vaccine:3 {vaccine}, antigen: {antigen}")
                    # print(f"uncovered demand for {antigen}: {uncovered_demand[antigen]}")
                    #transfer uncovered demand to next year

                print("^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^")
                inventory_DF_MCV.loc[inventory_DF_MCV.iloc[:, 0] == vaccine, 'Amount'] = remaining_inventory[vaccine]
                print(f"Adjusted Inventory of {vaccine} covering {antigen}: {inventory_DF_MCV}")

                print("Decrementing antigen demands")
                for ant in antigens:
                    print(f"antigen to decrement: {ant}")
                    current_demand = demand_80_MCV.loc[demand_80_MCV["antigen"] == ant, year].iloc[0]
                    if current_demand > 0:
                        print(f"pre-decrement {ant} demand: {demand_80_MCV.loc[demand_80_MCV['antigen'] == ant, year].iloc[0]}")
                        # print(f"From {ant} demand, reducing demand for year {year} for antigen {ant} by {decrement}")
                        # Update the demand for the antigen by subtracting the decrement in a single step
                        demand_80_MCV.loc[demand_80_MCV["antigen"] == ant, year] = current_demand - decrement
                        print(f"post decrement {ant} demand: {demand_80_MCV.loc[demand_80_MCV['antigen'] == ant, year].iloc[0]}")
            else:
                print(f"{antigen} not in {vaccine}")


        print(f"{len(antigen_counts)} antigens entered, only {least_covered_antigens} remain!")

    #update any uncovered demand, to next year. add uncovered demand to dosses_missed dict
    if 'uncovered_demand' in locals(): # Check if the variable exists
        while uncovered_demand:
            top = uncovered_demand.popitem()
            top_antigen = top[0]
            doses_missed = top[1]
            # print(f"{doses_missed} doses missed for {top_antigen}")
            demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == top_antigen, demand_80_MCV.columns[year+1]] += doses_missed
            missed_doses_MCV.loc[missed_doses_MCV.iloc[:, 0] == top_antigen, missed_doses_MCV.columns[1]] += doses_missed
            ###################################################################################################################
            #Need to add logic here to update interim_demand_DF[year +1]  with any uncovered demand
            interim_demand_DF.loc[interim_demand_DF.iloc[:,0]==top_antigen]
    else:
        print("no uncovered demand this year")
    
    #check ratio for supply/demand.
    #check at end of year for math reasons. if ratio is less than 1, schedule tender, perform search for vaccines, add inventory
    # print(f"Checking ratio of supply to demand for antigens for year {year + 1}!")
    # print()
    #pulls the current ratio of supply and demand. returns ratio_DF and antigen coverage DF
    ratio_DF_MCV, coverage_df = calculate_coverage_and_ratios(inventory_DF_MCV, demand_80_MCV, V_a, year+1)

    ratio_DF_MCV = ratio_DF_MCV.sort_values(by='antigen', key=lambda x: x.map(antigen_counts), ascending=True)

    for index, row in ratio_DF_MCV.iterrows():
        if row['Ratio'] < 1.0: #create three year tender
            #update:
            total_price += tender_cost
            # print(f"Ratio: {round(row.loc['Ratio'],2)}")
            print(f"Generating Tender for {row.loc['antigen']}")
            #append F schedule for curreny year +1 to current year +1 + tender_length
            new_row = {'Antigen': row.loc['antigen'], 'Starting': year + 1, 'Ending': year + tender_length }
            tender_schedules_MCV = pd.concat([tender_schedules_MCV, pd.DataFrame([new_row])], ignore_index=True)
            for time_period in range(1,tender_length+1):
                print("*^*^*^*^*^ TENDERING *^*^*^*^*^*^*^*^*^")
                print(f"TENDER PLANNING FOR YEAR: {year + time_period}")
                #inventory search
                price_list = get_manufacturer_vaccine_price(row.loc['antigen'], price_data, V_a, P_v, year + 1)
                # print(f"Lowest Price Information: {price_list}")
                # print(f"Pre fulfillment inventory: {inventory_DF_MCV}")
                # print("")
                # print(f"pre capacity_data: {capacity_data.loc[capacity_data['Manufacturer']==maunfacturer, year]}")
                result = fulfill_demand(price_list, row.loc['antigen'], interim_demand_DF, capacity_data, inventory_DF_MCV, year+time_period, total_price, A_v)
                # total_price += result['Total_Price']           
                print(result)
                # print(f"Post fulfillment inventory: {inventory_DF_MCV}")
                # print("")
                # print(f"post capacity_data: {capacity_data.loc[capacity_data['Manufacturer']==maunfacturer, year]}")

        else: #ratio greater than 1
            print(f"Ratio: {round(row.loc['Ratio'],2)}")
            print(f"Supply >= demand for {row.loc['antigen']}")



********************HAPPY NEW YEAR****************************
******************Reticulating splines...**************************
Year: 1
least covered antigens: ['Mumps', 'HPV', 'Rotavirus', 'PCV', 'Rubella', 'Measles', 'Hepatitis_B', 'Polio', 'Hib', 'Pertussis', 'Diphtheria', 'Tetanus']
###############################################################
serving antigen Mumps
Initial Demand: 26052200 for Mumps
Initial Inventory of vaccine covering Mumps:       Vaccine        Amount
0       Penta  1.370105e+08
1         OPV  2.837175e+08
2         IPV  9.879500e+06
3         PCV  1.088040e+08
4           M  2.575814e+08
5          MR  8.375001e+08
6         MMR  7.846400e+07
7          TT  1.000000e+07
8        HepB  1.000000e+07
9         Hib  1.000000e+07
10         DT  1.000000e+07
11         Td  1.438560e+08
12       DTwP  1.000000e+07
13   DTwP-Hib  1.000000e+07
14       Hexa  4.371450e+07
15        HPV  9.216212e+06
16  Rotavirus  6.745664e+07
A_v items are: dict_items([('M', ['Meas

C:\Users\nicho\AppData\Local\Temp\ipykernel_11680\184641932.py:54: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '9771988.14074928' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  demand_80_MCV.loc[demand_80_MCV["antigen"] == ant, year] = current_demand - decrement
C:\Users\nicho\AppData\Local\Temp\ipykernel_11680\184641932.py:69: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[2.41444462e+08]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == top_antigen, demand_80_MCV.columns[year+1]] += doses_missed


Generating Tender for HPV


NameError: name 'tender_schedules_MCV' is not defined

In [13]:
demand_80_MCV.loc[demand_80_MCV["antigen"] == 'Mumps', 1]

12    0
Name: 1, dtype: int64

In [29]:
demand_80_MCV

,antigen,1,2,3,4,5,6,7,8,9,10,11
10,Measles,0,0,0,0,0,0,0,0,0,0,0.1
12,Mumps,0,0,0,0,0,0,0,0,0,0,0.1
20,Rubella,0,0,0,0,0,0,0,0,0,0,0.1


In [30]:
tender_schedules_MCV

,Antigen,Starting,Ending
0,Measles,1,3
1,Mumps,1,3
2,Rubella,1,3
3,Mumps,4,6
4,Rubella,4,6
5,Measles,4,6
6,Mumps,7,9
7,Rubella,7,9
8,Measles,7,9
9,Mumps,10,12


In [16]:
inventory_DF_MCV


,Vaccine,Amount
4,M,0.0
5,MR,0.0
6,MMR,0.0
